In [40]:
import os
import sys
from datetime import datetime
from shapely.geometry import box
import numpy as np
import matplotlib.pyplot as plt

import os

# Load environment variables from .env file
load_dotenv()

# Add project path
project_path = os.getenv('PROJECT_ROOT')
if project_path not in sys.path:
    sys.path.insert(0, project_path)

# Set credentials
os.environ["COPERNICUS_USER"] = os.getenv('COPERNICUS_USER')
os.environ["COPERNICUS_PASSWORD"] = os.getenv('COPERNICUS_PASSWORD')

from src.data_acquisition.sources.sentinel_api import CustomSentinelAPI

def test_search_and_download():
    """Test searching and downloading Sentinel data"""
    try:
        # Initialize API with new endpoint
        print("Initializing API...")
        api = CustomSentinelAPI(
            api_url="https://apihub.copernicus.eu/apihub/"
        )
        print("API initialized successfully!")

        # Define search parameters
        bbox = (-122.5185, 37.7021, -122.3555, 37.8198)
        
        # Format dates in the correct format (YYYYMMDD)
        start_date = "20240101"  # Changed format
        end_date = "20240201"    # Changed format
        
        print("\nSearching for data...")
        print(f"Area: San Francisco Bay")
        print(f"Date range: {start_date} to {end_date}")
        print(f"Cloud cover limit: 20%")
        
        result = api.search_and_download(
            bbox=bbox,
            start_date=start_date,
            end_date=end_date,
            cloud_cover=20.0
        )
        
        if result:
            print("\nData retrieved successfully!")
            print(f"Data shape: {result['data'].shape}")
            print("\nMetadata:")
            for key, value in result['metadata'].items():
                print(f"{key}: {value}")
            
            # Visualize the data
            visualize_data(result)
        else:
            print("\nNo data found for the specified criteria")
            
        return result
        
    except Exception as e:
        print(f"\nError occurred: {type(e).__name__}")
        print(f"Error message: {str(e)}")
        print("\nNote: If you're getting authentication errors, you may need to:")
        print("1. Register at https://dataspace.copernicus.eu/")
        print("2. Update your credentials")
        print("3. Accept the terms and conditions on the website")
        import traceback
        traceback.print_exc()
        return None

def test_alternative_endpoint():
    """Test with alternative endpoint"""
    try:
        print("Testing alternative endpoint...")
        api = CustomSentinelAPI(
            api_url="https://dataspace.copernicus.eu/"
        )
        
        # Simple query to test connection
        bbox = (-122.5185, 37.7021, -122.3555, 37.8198)
        footprint = box(*bbox)
        
        # Use correct date format
        products = api.api.query(
            area=footprint.wkt,
            date=('20240101', '20240201'),  # Changed format
            platformname='Sentinel-2',
            cloudcoverpercentage=(0, 20)
        )
        
        print(f"Found {len(products)} products")
        return products
        
    except Exception as e:
        print(f"Alternative endpoint error: {e}")
        return None

def visualize_data(result):
    """Visualize the downloaded Sentinel data"""
    data = result['data']
    
    plt.figure(figsize=(15, 5))
    
    # RGB composite
    rgb = np.stack([
        data[2],  # Red (band 4)
        data[1],  # Green (band 3)
        data[0]   # Blue (band 2)
    ], axis=-1)
    
    # Normalize for visualization
    rgb = np.clip(rgb / 3000, 0, 1)
    
    plt.subplot(131)
    plt.imshow(rgb)
    plt.title("RGB Composite")
    plt.axis('off')
    
    # NDVI
    nir = data[3]  # Band 8 (NIR)
    red = data[2]  # Band 4 (Red)
    ndvi = (nir - red) / (nir + red)
    
    plt.subplot(132)
    plt.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
    plt.colorbar(label='NDVI')
    plt.title("NDVI")
    plt.axis('off')
    
    # False color composite
    false_color = np.stack([
        data[3],  # NIR
        data[2],  # Red
        data[1]   # Green
    ], axis=-1)
    false_color = np.clip(false_color / 3000, 0, 1)
    
    plt.subplot(133)
    plt.imshow(false_color)
    plt.title("False Color (NIR)")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    print("Testing Sentinel API functionality...\n")
    
    print("1. Testing primary endpoint...")
    result = test_search_and_download()
    
    if not result:
        print("\n2. Testing alternative endpoint...")
        result = test_alternative_endpoint()

Testing Sentinel API functionality...

1. Testing primary endpoint...
Initializing API...
API initialized successfully!

Searching for data...
Area: San Francisco Bay
Date range: 20240101 to 20240201
Cloud cover limit: 20%

Error occurred: ConnectTimeout
Error message: HTTPSConnectionPool(host='apihub.copernicus.eu', port=443): Max retries exceeded with url: /apihub/search?format=json&rows=100&start=0&q=beginPosition%3A%5B%222024-01-01T00%3A00%3A00Z%22+TO+%222024-02-01T00%3A00%3A00Z%22%5D+cloudcoverpercentage%3A%5B%220%22+TO+%2220.0%22%5D+platformname%3A%22Sentinel-2%22+producttype%3A%22S2MSI2A%22+footprint%3A%22Intersects%28POLYGON+%28%28-122.3555+37.7021%2C+-122.3555+37.8198%2C+-122.5185+37.8198%2C+-122.5185+37.7021%2C+-122.3555+37.7021%29%29%29%22 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7bf245f5b760>, 'Connection to apihub.copernicus.eu timed out. (connect timeout=None)'))

Note: If you're getting authentication errors, you may need to:
1. Regi

Traceback (most recent call last):
  File "/home/jaya/myenv/lib/python3.10/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
  File "/home/jaya/myenv/lib/python3.10/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/home/jaya/myenv/lib/python3.10/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
TimeoutError: [Errno 110] Connection timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/jaya/myenv/lib/python3.10/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
  File "/home/jaya/myenv/lib/python3.10/site-packages/urllib3/connectionpool.py", line 488, in _make_request
    raise new_e
  File "/home/jaya/myenv/lib/python3.10/site-packages/urllib3/connectionpool.py", line 464, in _make_request
    self._validate_conn(conn)
  File 

Alternative endpoint error: HTTP status 404 Not Found: 
* [News](/news)
  * [Dashboard](/copernicus-data-space-ecosystem-dashboard)
  * [Cases](https://dataspace.copernicus.eu/cases)
  * [Events](/events)
  * [Gallery](/gallery)
  * [Videos](/videos)
  * [About](/about)

[ ](/ "Home - Copernicus Data Space Ecosystem")

  * [Explore data](/explore-data/data-collections)
    * [Sentinel data](/explore-data/data-collections/sentinel-data)
    * [Copernicus Contributing Missions](/explore-data/data-collections/copernicus-contributing-missions)
    * [Federated data sets](/explore-data/data-collections/federated-data-sets)
    * [Complementary data](https://dataspace.copernicus.eu/complementary-data)
  * [Analyse Data](/analyse)
    * [APIs](/analyse/apis)
    * [Data Workspace](/analyse/data-workspace)
    * [Traceability](/analyse/traceability)
    * [JupyterLab](/analyse/jupyterlab)
    * [openEO](/analyse/openeo)
    * [Sentinel Hub](https://shapps.dataspace.copernicus.eu/dashboard/#/)


In [41]:
import os
import sys
from datetime import datetime
from shapely.geometry import box
import numpy as np
import matplotlib.pyplot as plt

# Add project path
project_path = '/home/jaya/synthetic-satellite'
if project_path not in sys.path:
    sys.path.insert(0, project_path)

# Set credentials
os.environ["COPERNICUS_USER"] = "jaya2424@gmail.com"
os.environ["COPERNICUS_PASSWORD"] = "Vortx#12345678"

from src.data_acquisition.sources.sentinel_api import CustomSentinelAPI

def test_search_and_download():
    """Test searching and downloading Sentinel data"""
    try:
        # Initialize API with SciHub endpoint
        print("Initializing API...")
        api = CustomSentinelAPI(
            api_url="https://scihub.copernicus.eu/dhus"  # Original SciHub endpoint
        )
        print("API initialized successfully!")

        # Define search parameters
        bbox = (-122.5185, 37.7021, -122.3555, 37.8198)  # San Francisco Bay Area

        # Dates in YYYYMMDD format
        start_date = "20240101"
        end_date = "20240201"

        print("\nSearching for data...")
        print(f"Area: San Francisco Bay")
        print(f"Date range: {start_date} to {end_date}")
        print(f"Cloud cover limit: 20%")

        result = api.search_and_download(
            bbox=bbox,
            start_date=start_date,
            end_date=end_date,
            cloud_cover=20.0
        )

        if result:
            print("\nData retrieved successfully!")
            print(f"Data shape: {result['data'].shape}")
            print("\nMetadata:")
            for key, value in result['metadata'].items():
                print(f"{key}: {value}")

            # Visualize the data
            visualize_data(result)
        else:
            print("\nNo data found for the specified criteria")

        return result

    except Exception as e:
        print(f"\nError occurred: {type(e).__name__}")
        print(f"Error message: {str(e)}")
        print("\nNote: If you're getting authentication errors, you may need to:")
        print("1. Register at https://scihub.copernicus.eu/dhus")
        print("2. Update your credentials")
        print("3. Accept the terms and conditions on the website")
        import traceback
        traceback.print_exc()
        return None

def visualize_data(result):
    """Visualize the downloaded Sentinel data"""
    data = result['data']

    plt.figure(figsize=(15, 5))

    # RGB composite
    rgb = np.stack([
        data[2],  # Red (band 4)
        data[1],  # Green (band 3)
        data[0]   # Blue (band 2)
    ], axis=-1)

    # Normalize for visualization
    rgb = np.clip(rgb / 3000, 0, 1)

    plt.subplot(131)
    plt.imshow(rgb)
    plt.title("RGB Composite")
    plt.axis('off')

    # NDVI
    nir = data[3]  # Band 8 (NIR)
    red = data[2]  # Band 4 (Red)
    ndvi = (nir - red) / (nir + red)

    plt.subplot(132)
    plt.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
    plt.colorbar(label='NDVI')
    plt.title("NDVI")
    plt.axis('off')

    # False color composite
    false_color = np.stack([
        data[3],  # NIR
        data[2],  # Red
        data[1]   # Green
    ], axis=-1)
    false_color = np.clip(false_color / 3000, 0, 1)

    plt.subplot(133)
    plt.imshow(false_color)
    plt.title("False Color (NIR)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

def test_product_info():
    """Test retrieving product information"""
    try:
        api = CustomSentinelAPI(
            api_url="https://scihub.copernicus.eu/dhus"
        )

        # Search for products first
        bbox = (-122.5185, 37.7021, -122.3555, 37.8198)
        products = api.api.query(
            area=box(*bbox).wkt,
            date=('20240101', '20240201'),
            platformname='Sentinel-2',
            cloudcoverpercentage=(0, 20)
        )

        if products:
            # Get first product ID
            product_id = list(products.keys())[0]

            # Get product info
            info = api.get_product_info(product_id)
            print("\nProduct Information:")
            for key, value in info.items():
                print(f"{key}: {value}")
        else:
            print("No products found")

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    print("Testing Sentinel API functionality...\n")

    print("1. Testing search and download...")
    result = test_search_and_download()

    print("\n2. Testing product info retrieval...")
    test_product_info()

Testing Sentinel API functionality...

1. Testing search and download...
Initializing API...
API initialized successfully!

Searching for data...
Area: San Francisco Bay
Date range: 20240101 to 20240201
Cloud cover limit: 20%

Error occurred: ServerError
Error message: HTTP status 200 OK: 
The Sentinels Scientific Data Hub

# Copernicus Sentinel Data is now available on the Copernicus Data Space
Ecosystem

[https://dataspace.copernicus.eu](https://dataspace.copernicus.eu/)

Note: If you're getting authentication errors, you may need to:
1. Register at https://scihub.copernicus.eu/dhus
2. Update your credentials
3. Accept the terms and conditions on the website

2. Testing product info retrieval...


Traceback (most recent call last):
  File "/home/jaya/myenv/lib/python3.10/site-packages/requests/models.py", line 974, in json
    return complexjson.loads(self.text, **kwargs)
  File "/usr/lib/python3.10/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "/usr/lib/python3.10/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "/usr/lib/python3.10/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/jaya/myenv/lib/python3.10/site-packages/sentinelsat/sentinel.py", line 1025, in _check_scihub_response
    response.json()
  File "/home/jaya/myenv/lib/python3.10/site-packages/requests/models.py", line 978, in json
    raise RequestsJSONDecodeError(e.msg, e.doc, e.po

Error: HTTP status 200 OK: 
The Sentinels Scientific Data Hub

# Copernicus Sentinel Data is now available on the Copernicus Data Space
Ecosystem

[https://dataspace.copernicus.eu](https://dataspace.copernicus.eu/)
